# ML Practice – korrigierte Version

CSV-Pfad und Zielspalte in Config prüfen, dann alle Zellen von oben ausführen.

Abhängigkeiten: numpy, pandas, matplotlib, seaborn, scipy, scikit-learn. Optional: xgboost; imbalanced-learn nur für Oversample=True.

UseGrid=True und UseCv=True verwenden verschachtelte CV und benötigen mehr Rechenzeit. CV-test bedeutet Validierungsfold; *_test_f1 bezeichnet den separaten Holdout. Den Holdout nicht wiederholt zur Modellauswahl verwenden.

DataTransform bleibt absichtlich dein unveränderter Platzhalter: BASE und TRANSFORMED sind bis zu eigenen Transformationen gleichwertig. Es werden keine Feature-Engineering-Regeln erfunden.

Oversample ersetzt den bisher fehlerhaften SMOTE-Schalter durch RandomOverSampler (reale Trainingszeilen duplizieren). Standard: ausgeschaltet.

Die gespeicherte Originalausgabe zeigte nur 156 Titanic-Zeilen. Prüfe, ob das der beabsichtigte Datensatz ist.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_validate, RandomizedSearchCV, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.utils.validation import check_is_fitted
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, make_scorer
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

In [ ]:
class Config():

    def __init__(self):
        self.seed = 1234
        self.target = "Survived"
        self.test_size = 0.2
        self.csv_path = "/run/media/drdrakken/Elements_8TB/Sonstiges/Programmieren/Machine Learning/ProjectFolder/A1/Utillitys/utils/csvs/titanic.csv"
        self.k_splits = 5
        self.verbose = 0
        self.test_iterations = self.k_splits

CONFIG = Config()     

In [ ]:
def data_class(csv_file=None):
    path = CONFIG.csv_path if csv_file is None else csv_file
    return pd.read_csv(path)

In [ ]:
class GetData:
    def __init__(self, csv_file=None):
        self.data = data_class(csv_file)
        if CONFIG.target not in self.data:
            raise ValueError(f"Zielspalte fehlt: {CONFIG.target}")
        if self.data[CONFIG.target].isna().any():
            raise ValueError("Die Zielspalte enthält fehlende Werte.")
        if not set(self.data[CONFIG.target].unique()).issubset({0, 1}):
            raise ValueError("Dieses Notebook erwartet eine binäre Zielspalte mit 0 und 1.")
        self.x = self.data.drop(columns=[CONFIG.target])
        self.y = self.data[CONFIG.target]
        self.numerical_data = self.x.select_dtypes(include=np.number).columns
        self.categorical_data = self.x.select_dtypes(exclude=np.number).columns

DATA = GetData()

In [ ]:
class Visualize:
    def __init__(self, data=None):
        self.data = DATA if data is None else data

    def iqr(self, iqr_col=None):
        if iqr_col is not None:
            values = self.data.data[iqr_col]
            q1, q3 = values.quantile([0.25, 0.75])
            return q1, q3, q3 - q1

    def skew(self, skew_col=None):
        if skew_col is not None:
            value = self.data.data[skew_col].skew()
            print(f"Skewness of {skew_col}: {value}")
            return value

    def plot(self):
        df = self.data.data
        for vis in self.data.numerical_data:
            fig, axes = plt.subplots(3, 1, figsize=(10, 10))
            q1, q3, _ = self.iqr(vis)
            mean = df[vis].mean()
            self.skew(vis)
            sns.histplot(data=df, x=vis, ax=axes[0])
            sns.boxplot(data=df, x=vis, ax=axes[1])
            sns.scatterplot(data=df, x=vis, y=CONFIG.target, ax=axes[2])
            for ax, title in zip(axes, ('Histogramm', 'Boxplot', 'Zielvergleich')):
                ax.axvline(q1, color='green', label='Q1')
                ax.axvline(q3, color='red', label='Q3')
                ax.axvline(mean, color='orange', label='Mittelwert')
                ax.set_title(f'{title}: {vis}')
            axes[0].legend()
            fig.tight_layout()
            plt.show()

In [ ]:
class DataTransform(BaseEstimator , TransformerMixin):

    def fit(self , X , y = None):
        return self

    def transform(self , X , y = None):
        x = X.copy()
        x = self.transform_data(x)
        return x

    def transform_data(self , X , y = None):
        x = X.copy()

        return x
    

In [ ]:
class Preprocess(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        numerical = X.select_dtypes(include=np.number).columns.tolist()
        categorical = X.select_dtypes(exclude=np.number).columns.tolist()
        self.preprocessor_ = ColumnTransformer([
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='mean', keep_empty_features=True)),
                ('scaler', MinMaxScaler()),
            ]), numerical),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
                ('encoder', OneHotEncoder(handle_unknown='ignore')),
            ]), categorical),
        ])
        self.preprocessor_.fit(X, y)
        return self

    def transform(self, X):
        check_is_fitted(self, 'preprocessor_')
        return self.preprocessor_.transform(X)

In [ ]:
def build_pipeline(model, Transform=False, Oversample=False):
    steps = []
    if Transform:
        steps.append(('TransformedData', DataTransform()))
    steps.append(('Base', Preprocess()))
    if Oversample:
        # RandomOverSampler duplicates real rows: no fractional one-hot categories.
        # It is fitted only on each training fold through the imblearn pipeline.
        from imblearn.over_sampling import RandomOverSampler
        from imblearn.pipeline import Pipeline as SamplingPipeline
        steps.append(('Oversampler', RandomOverSampler(random_state=CONFIG.seed)))
        steps.append(('Model', clone(model)))
        return SamplingPipeline(steps)
    steps.append(('Model', clone(model)))
    return Pipeline(steps)

In [ ]:
def ParamGrid():

    return {

        "LogisticRegression": {
            "Model__C": scipy.stats.loguniform(1e-4, 1e2),
            "Model__solver": ["lbfgs"],
        },

        "DecisionTreeClassifier": {
            "Model__criterion": ["gini", "entropy"],
            "Model__max_depth": [None, 5, 10, 20],
            "Model__min_samples_split": scipy.stats.randint(2, 20),
            "Model__min_samples_leaf": scipy.stats.randint(1, 10),
        },

        "RandomForestClassifier": {
            "Model__n_estimators": scipy.stats.randint(100, 500),
            "Model__max_depth": scipy.stats.randint(5, 40),
            "Model__min_samples_split": scipy.stats.randint(2, 20),
            "Model__min_samples_leaf": scipy.stats.randint(1, 10),
            "Model__max_features": ["sqrt", "log2"],
            "Model__bootstrap": [True, False],
        }, 

        "LinearSVC": {
            "Model__C": scipy.stats.loguniform(1e-4, 1e2),
            "Model__loss": ["hinge", "squared_hinge"],
            "Model__dual": [True],
            "Model__max_iter": [5000, 10000],
        },

        "XGBClassifier": {
            "Model__n_estimators": scipy.stats.randint(100, 500),
            "Model__learning_rate": scipy.stats.loguniform(1e-3, 0.3),
            "Model__max_depth": scipy.stats.randint(3, 10),
            "Model__subsample": scipy.stats.uniform(0.6, 0.4),
            "Model__colsample_bytree": scipy.stats.uniform(0.6, 0.4),
            "Model__gamma": scipy.stats.uniform(0, 5),
            "Model__min_child_weight": scipy.stats.randint(1, 10),
        },

    }

In [ ]:
def scoring_rules():
    return {
        'accuracy': 'accuracy',
        'precision': make_scorer(precision_score, zero_division=0),
        'recall': make_scorer(recall_score, zero_division=0),
        'f1': make_scorer(f1_score, zero_division=0),
    }

def make_cv():
    return StratifiedKFold(n_splits=CONFIG.k_splits, shuffle=True, random_state=CONFIG.seed)

def custom_cv(model, X_train, y_train):
    return cross_validate(
        estimator=model, X=X_train, y=y_train,
        scoring=scoring_rules(), cv=make_cv(),
        verbose=CONFIG.verbose, n_jobs=-1,
        return_train_score=True, error_score='raise',
    )

In [ ]:
def custom_grid(model, X_train=None, y_train=None, custom_param_grid=None):
    search = RandomizedSearchCV(
        estimator=clone(model), param_distributions=custom_param_grid,
        n_iter=CONFIG.test_iterations, scoring=scoring_rules(), refit='f1',
        cv=make_cv(), random_state=CONFIG.seed,
        verbose=CONFIG.verbose, n_jobs=1,
        return_train_score=True, error_score='raise',
    )
    if X_train is not None and y_train is not None:
        search.fit(X_train, y_train)
    return search

In [ ]:
def model_varianz():
    models = {
        'LogisticRegression': LogisticRegression(random_state=CONFIG.seed, max_iter=5000),
        'LinearSVC': LinearSVC(random_state=CONFIG.seed, max_iter=10000),
        'DecisionTreeClassifier': DecisionTreeClassifier(random_state=CONFIG.seed),
        'RandomForestClassifier': RandomForestClassifier(random_state=CONFIG.seed, n_jobs=1),
    }
    if XGBClassifier is not None:
        models['XGBClassifier'] = XGBClassifier(
            random_state=CONFIG.seed, n_jobs=1, eval_metric='logloss'
        )
    else:
        print('xgboost ist nicht installiert; XGBClassifier wird übersprungen.')
    return models

In [ ]:
def data_split():
    X_train , X_test , y_train , y_test = train_test_split(DATA.x , 
                                                        DATA.y,
                                                        test_size= CONFIG.test_size,
                                                        random_state=CONFIG.seed,
                                                        stratify=DATA.y)
    return X_train , X_test , y_train , y_test 

In [ ]:
class Benchmark:
    def __init__(self, UseGrid=False, UseCv=False, Oversample=False):
        self.X_train, self.X_test, self.y_train, self.y_test = data_split()
        self.results = {}
        self.fitted_models = {}
        self.use_grid = UseGrid
        self.use_cv = UseCv
        self.oversample = Oversample
        self.benchmark()

    def benchmark(self):
        for name, model in model_varianz().items():
            results = {'MODEL': name}
            self.fitted_models[name] = {}
            for prefix, transformed in (('BASE', False), ('TRANSFORMED', True)):
                print(f'{name} / {prefix}')
                pipe = build_pipeline(model, Transform=transformed, Oversample=self.oversample)
                if self.use_grid:
                    # Unfitted search as the outer CV estimator -> nested CV.
                    estimator = custom_grid(pipe, custom_param_grid=ParamGrid()[name])
                else:
                    estimator = pipe

                if self.use_cv:
                    cv = custom_cv(estimator, self.X_train, self.y_train)
                    for metric in ('f1', 'precision', 'recall', 'accuracy'):
                        for method in ('train', 'test'):
                            values = cv[f'{method}_{metric}']
                            results[f'{prefix}_{method}__CV__{metric}'] = values.mean()
                            results[f'{prefix}_{method}__STD__{metric}'] = values.std()

                # Final refit on all training rows; the holdout stays out of search/CV.
                fitted = clone(estimator).fit(self.X_train, self.y_train)
                final_model = fitted.best_estimator_ if self.use_grid else fitted
                self.fitted_models[name][prefix] = final_model
                if self.use_grid:
                    self.get_test_set(prefix=prefix, results=results, grid=fitted)
                self.eval_model(prefix, final_model, results)
            self.results[name] = results

        if self.use_cv:
            self.plot_cv()
        self.summary = pd.DataFrame(self.results).T

    def eval_model(self, prefix=None, fittet_obj=None, results=None):
        if prefix is None or fittet_obj is None or results is None:
            return
        for method, X, y in (
            ('train', self.X_train, self.y_train),
            ('test', self.X_test, self.y_test),
        ):
            predicted = fittet_obj.predict(X)
            results[f'{prefix}_{method}_f1'] = f1_score(y, predicted, zero_division=0)
            results[f'{prefix}_{method}_precision'] = precision_score(y, predicted, zero_division=0)
            results[f'{prefix}_{method}_recall'] = recall_score(y, predicted, zero_division=0)
            results[f'{prefix}_{method}_accuracy'] = accuracy_score(y, predicted)

    def get_test_set(self, prefix=None, fittet_obj=None, results=None, grid=None):
        # Historical method name: stores search results, not test data.
        if prefix is not None and results is not None and grid is not None:
            results[f'{prefix}_best_estimator'] = grid.best_estimator_
            results[f'{prefix}_best_params'] = grid.best_params_
            results[f'{prefix}_inner_best_cv_f1'] = grid.best_score_

    def plot_cv(self, metric="f1", method="test"):

        model_names = []
        base_scores = []
        base_stds = []
        transformed_scores = []
        transformed_stds = []

        for model_name, result in self.results.items():
            base_key = f"BASE_{method}__CV__{metric}"
            base_std_key = f"BASE_{method}__STD__{metric}"

            transformed_key = f"TRANSFORMED_{method}__CV__{metric}"
            transformed_std_key = f"TRANSFORMED_{method}__STD__{metric}"

            # Modelle ohne gespeicherte CV-Ergebnisse überspringen
            if base_key not in result or transformed_key not in result:
                continue

            model_names.append(model_name)
            base_scores.append(result[base_key])
            base_stds.append(result.get(base_std_key, 0))
            transformed_scores.append(result[transformed_key])
            transformed_stds.append(result.get(transformed_std_key, 0))

        if not model_names:
            print("Keine CV-Ergebnisse gefunden.")
            return

        x = np.arange(len(model_names))
        width = 0.36

        fig, ax = plt.subplots(figsize=(10, 6))

        ax.bar(
            x - width / 2,
            base_scores,
            width,
            yerr=base_stds,
            capsize=5,
            label="BASE"
        )

        ax.bar(
            x + width / 2,
            transformed_scores,
            width,
            yerr=transformed_stds,
            capsize=5,
            label="TRANSFORMED"
        )

        ax.set_title(f"{metric.upper()}-Score ({method} CV)")
        ax.set_ylabel(metric.upper())
        ax.set_xticks(x)
        ax.set_xticklabels(model_names, rotation=30, ha="right")
        ax.set_ylim(0, 1)
        ax.legend()
        ax.grid(axis="y", alpha=0.3)

        fig.tight_layout()
        plt.show()

In [ ]:
benchmark = Benchmark(UseGrid=True, UseCv=True, Oversample=False)
# Nur numerische Auswertungen anzeigen; Modelle liegen in benchmark.fitted_models.
benchmark.summary.select_dtypes(include='number')